# Granger Causality & OU Forecasting — Pairs Trading Add-On

This notebook adds two analyses on top of the main pairs trading project:

1. **Granger Causality Tests** — does one asset in each pair *lead* the other? Identifies price leaders vs followers and informs signal timing.

2. **OU Forecasting with Train/Test Validation** — fit the Ornstein-Uhlenbeck model on all data *except* the last 6 months, then test whether the model correctly predicted what the spread did in those final 6 months (Feb 2023 → Aug 2023).

**Dataset:** `commodity_futures.csv` — must be in the same directory.

**Selected pairs** (same as main notebook):
- WTI Crude / Brent Crude
- Low Sulphur Gas Oil / ULS Diesel
- Soybeans / Soybean Meal
- Corn / Wheat


## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

plt.rcParams.update({
    'figure.figsize': (14, 5), 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11
})
np.random.seed(42)

PAIRS = [
    ('WTI CRUDE',           'BRENT CRUDE',    'Energy — Crude Oil'),
    ('LOW SULPHUR GAS OIL', 'ULS DIESEL',     'Energy — Refined Products'),
    ('SOYBEANS',            'SOYBEAN MEAL',   'Agricultural — Crush Spread'),
    ('CORN',                'WHEAT',          'Agricultural — Feed Grains'),
]

print('Libraries loaded.')


## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('commodity_futures.csv', parse_dates=['Date'], index_col='Date')
df_clean = df.drop(columns=['GASOLINE'], errors='ignore').ffill().bfill()

# Compute static hedge ratio and spread for each pair (using full data)
# β estimated on full sample — same as main notebook
pair_info = {}
for a1, a2, label in PAIRS:
    p = df_clean[[a1, a2]].dropna()
    X = sm.add_constant(p[a1])
    beta = sm.OLS(p[a2], X).fit().params[a1]
    spread = p[a2] - beta * p[a1]
    pair_info[(a1, a2)] = {'label': label, 'beta': round(beta, 4),
                           'spread': spread, 'prices': p}
    print(f'[{label}]  β = {beta:.4f}  '
          f'Spread mean = {spread.mean():.2f}  std = {spread.std():.2f}')

# Train/test cutoff: everything EXCEPT last 6 months
max_date   = df_clean.index.max()
CUTOFF     = max_date - pd.DateOffset(months=6)
print(f'\nFull data:  {df_clean.index.min().date()} → {max_date.date()}')
print(f'Train ends: {CUTOFF.date()}')
print(f'Test start: {CUTOFF.date()}  ({len(df_clean.loc[CUTOFF:]) - 1} trading days)')


## PART 1: Granger Causality Tests

## What Granger Causality Is

Granger causality does **not** mean true causation. It asks a specific, testable question:

> *Does knowing the past values of series A improve my ability to forecast series B, beyond what B's own past values already tell me?*

If yes, then A **Granger-causes** B. This identifies **price leadership** — which asset in the pair tends to move first, with the other following.

### The Test Mechanics

Two VAR models are compared for each direction:

**Restricted model** (B's own lags only):
```
B_t = α + Σ β_k B_{t-k} + ε_t
```

**Unrestricted model** (B's own lags + A's lags):
```
B_t = α + Σ β_k B_{t-k} + Σ γ_k A_{t-k} + ε_t
```

An F-test checks whether adding A's lags significantly reduces the residual variance. **H₀: A does NOT Granger-cause B** (all γ_k = 0). Reject H₀ (p < 0.05) → A leads B.

### Why This Matters for Pairs Trading

| Finding | Trading Implication |
|---|---|
| A one-way leads B | When A diverges from equilibrium, enter the trade — B will follow. Signal timing improves. |
| B one-way leads A | Reverse: B is the price setter. |
| Bidirectional | Both series adjust simultaneously — no timing advantage. |
| No causality | The cointegration works but contemporaneously — no predictive lead/lag relationship. |

### Important: Use Price *Differences*, Not Levels

We run Granger causality on **first differences** (daily returns) of each price series. Running it on price levels (which are I(1) non-stationary) would produce spurious results because non-stationary series violate the stationarity assumption of the F-test.


## 2. Run Granger Causality — All 4 Pairs, Lags 1, 2, 5, 10

In [ ]:
LAGS_TO_TEST = [1, 2, 5, 10]  # 1 day, 2 days, 1 week, 2 weeks

granger_results = {}

for a1, a2, label in PAIRS:
    p = pair_info[(a1, a2)]['prices'].copy()

    # First differences (daily returns) — required for stationarity
    dp = p.diff().dropna()

    results_pair = {}

    for direction, (y_col, x_col) in [('A→B', (a2, a1)), ('B→A', (a1, a2))]:
        data = dp[[y_col, x_col]]  # statsmodels: first col = Y (predicted), second = X (predictor)
        gc = grangercausalitytests(data, maxlag=max(LAGS_TO_TEST), verbose=False)
        lag_results = {}
        for lag in LAGS_TO_TEST:
            f_stat = gc[lag][0]['ssr_ftest'][0]
            p_val  = gc[lag][0]['ssr_ftest'][1]
            lag_results[lag] = {'f': round(f_stat, 3), 'p': round(p_val, 4)}
        results_pair[direction] = lag_results

    granger_results[(a1, a2)] = results_pair

print('Granger causality tests complete.')


## 3. Results Table — Formatted Output

In [ ]:
print(f'{'='*80}')
print(f'GRANGER CAUSALITY RESULTS  (p-values | * p<0.05  ** p<0.01  *** p<0.001)')
print(f'{'='*80}')

for a1, a2, label in PAIRS:
    res = granger_results[(a1, a2)]
    print(f'\n{label}: {a1} / {a2}')
    print(f'  Direction         Lag-1        Lag-2        Lag-5        Lag-10')
    print(f'  {'-'*65}')
    for direction, (y_col, x_col) in [('A→B  (X leads Y)', (a2, a1)),
                                       ('B→A  (Y leads X)', (a1, a2))]:
        short_dir = f'{a1}→{a2}' if direction.startswith('A') else f'{a2}→{a1}'
        row = f'  {short_dir[:28]:<28}'
        for lag in LAGS_TO_TEST:
            p = res[direction[:3]][lag]['p']
            stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '  '))
            row += f'  {p:.4f}{stars}'
        print(row)

    # Interpret
    fwd_sig = any(res['A→B'][l]['p'] < 0.05 for l in LAGS_TO_TEST)
    bwd_sig = any(res['B→A'][l]['p'] < 0.05 for l in LAGS_TO_TEST)
    if fwd_sig and bwd_sig:
        interp = f'↔ Bidirectional — both series adjust to each other'
    elif fwd_sig:
        interp = f'→ {a1} is the PRICE LEADER — {a2} follows'
    elif bwd_sig:
        interp = f'→ {a2} is the PRICE LEADER — {a1} follows'
    else:
        interp = '— No Granger causality — contemporaneous comovement only'
    print(f'  Interpretation: {interp}')


## 4. Visualisation — F-Statistics Heatmap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, (a1, a2, label) in zip(axes, PAIRS):
    res = granger_results[(a1, a2)]
    directions = [f'{a1}\n→{a2}', f'{a2}\n→{a1}']
    matrix = np.zeros((2, len(LAGS_TO_TEST)))
    for i, direction in enumerate(['A→B', 'B→A']):
        for j, lag in enumerate(LAGS_TO_TEST):
            matrix[i, j] = res[direction][lag]['p']

    mat_df = pd.DataFrame(matrix, index=directions,
                          columns=[f'Lag {l}' for l in LAGS_TO_TEST])

    # Colour by significance: green = significant, red = not
    sns.heatmap(mat_df, annot=True, fmt='.4f', ax=ax,
                cmap='RdYlGn_r', vmin=0, vmax=0.1,
                linewidths=0.5, cbar=False)
    ax.set_title(f'{label}\nGranger Causality p-values (green = significant, p<0.05)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Lag Window')

plt.suptitle('Granger Causality Tests — All Pairs\n'
             'Testing on first-differenced prices (stationary)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Minimum Significant Lag — Summary

In [ ]:
print('Price Leadership Summary')
print('='*60)

for a1, a2, label in PAIRS:
    res = granger_results[(a1, a2)]
    print(f'\n{label}')
    for direction, name in [('A→B', f'{a1} → {a2}'), ('B→A', f'{a2} → {a1}')]:
        min_sig_lag = None
        for lag in LAGS_TO_TEST:
            if res[direction][lag]['p'] < 0.05:
                min_sig_lag = lag
                break
        if min_sig_lag:
            p_min = res[direction][min_sig_lag]['p']
            f_min = res[direction][min_sig_lag]['f']
            print(f'  {name}: significant at lag {min_sig_lag} '
                  f'(F={f_min}, p={p_min:.4f}) ← LEADS')
        else:
            print(f'  {name}: NOT significant at any tested lag')

## 6. Granger Causality — Interpretation Guide

### What Each Result Means for Trading

**One-way causality (A → B only):**
- A is the **price leader** — it reacts to new information first
- B is the **price follower** — it adjusts with a delay of `lag` trading days
- Trading implication: when you see a large move in A that pushes the spread away from equilibrium, you can enter the pairs trade immediately, before B has had time to adjust. This gives you a slightly better entry price.

**Bidirectional causality (A → B and B → A):**
- Both series react to each other's past movements
- This is common for markets that are tightly linked and trade in overlapping time zones (e.g. WTI on NYMEX and Brent on ICE)
- No timing advantage — enter when the Z-score signals, as usual

**No causality:**
- The cointegration works through contemporaneous shocks (same-day) rather than lead-lag dynamics
- The relationship is real but not exploitable through timing

### Important Caveat — Granger Causality on Cointegrated Series

When two series are cointegrated, the correct way to model lead-lag dynamics is the **VECM error-correction term** (already in the main notebook), which captures both short-run lead-lag effects AND the long-run adjustment to equilibrium. Granger causality on first differences is a simpler, more interpretable complement but misses the long-run cointegration channel. The two approaches are complementary.


# PART 2: OU Forecasting with Train / Test Validation

## The Setup

We split the data into:
- **Training set:** January 2000 → February 2023 (~23 years, ~5,962 trading days)
- **Test set:** February 2023 → August 2023 (~6 months, ~130 trading days)

The OU model is fitted **entirely on training data**, then asked to forecast the spread for the next 130 days. We then overlay the **actual spread** from the test period to see how accurate the prediction was.

## Why This Is a Proper Validation

Unlike the full-history OU forecast (which is a forward-looking prediction into 2025), this is a **backtested validation** — we can directly compare the model's predictions against known outcomes. This gives us concrete accuracy metrics:

| Metric | What It Measures |
|---|---|
| **Coverage (50% CI)** | What % of actual spread values fell inside the 50% band? Should be ~50%. |
| **Coverage (90% CI)** | What % fell inside the 90% band? Should be ~90%. |
| **Median RMSE** | Root mean squared error of the median forecast path vs actual spread. |
| **Convergence accuracy** | Did the model correctly predict when/if the spread crossed θ? |

## How the Forecast Is Built

```
Step 1: Estimate β (hedge ratio) on training data only
Step 2: Build spread = Y - β*X on ALL data (to see full history)
Step 3: Fit OU parameters (κ, θ, σ) on training spread only
Step 4: S0 = last spread value in training period
Step 5: Simulate 5,000 OU paths forward for 130 days
Step 6: Compare simulated paths against actual test-period spread
```


## 7. OU Helper Functions

In [ ]:
def fit_ou(spread):
    """
    Estimate OU parameters from spread history using AR(1) OLS.
    Regress S_t on S_{t-1} to get φ and α, then convert to κ, θ, σ.
    """
    s    = spread.dropna()
    lag  = s.shift(1).dropna()
    curr = s.loc[lag.index]

    res   = sm.OLS(curr, sm.add_constant(lag)).fit()
    phi   = res.params.iloc[1]          # AR(1) coefficient
    alpha = res.params['const']         # intercept

    # Clamp phi to (0, 1) — must be mean-reverting
    phi = max(min(phi, 0.9999), 0.0001)

    kappa = -np.log(phi)                 # reversion speed
    theta = alpha / (1 - phi)            # long-run mean
    sigma = np.sqrt(res.mse_resid)       # daily volatility
    hl    = np.log(2) / kappa            # half-life in trading days

    return {'kappa': kappa, 'theta': theta, 'sigma': sigma,
            'half_life': hl, 'phi': phi, 'alpha': alpha}


def simulate_ou(params, S0, n_steps, n_paths=5000):
    """
    Simulate n_paths forward trajectories of the OU process.
    Discretised Euler-Maruyama: S_{t+1} = S_t + κ(θ - S_t) + σ*ε_t
    Returns array of shape (n_steps+1, n_paths).
    """
    kappa, theta, sigma = params['kappa'], params['theta'], params['sigma']
    paths      = np.zeros((n_steps + 1, n_paths))
    paths[0]   = S0
    shocks     = np.random.randn(n_steps, n_paths)  # all shocks pre-drawn
    for t in range(1, n_steps + 1):
        paths[t] = (paths[t-1]
                    + kappa * (theta - paths[t-1])
                    + sigma * shocks[t-1])
    return paths


def convergence_stats(paths, S0, theta, dates):
    """
    For each simulated path, find the first day the spread crosses theta.
    Returns crossing day indices and the corresponding dates.
    """
    direction   = 1 if S0 > theta else -1   # +1 = spread above mean, -1 = below
    crossings   = []
    for path in paths.T:  # iterate over paths (columns)
        if direction == 1:
            idx = np.where(path <= theta)[0]
        else:
            idx = np.where(path >= theta)[0]
        if len(idx):
            crossings.append(idx[0])

    pct_converge = len(crossings) / paths.shape[1] * 100
    if crossings:
        med_day  = int(np.median(crossings))
        p25_day  = int(np.percentile(crossings, 25))
        p75_day  = int(np.percentile(crossings, 75))
        med_date = dates[min(med_day, len(dates)-1)]
        p25_date = dates[min(p25_day, len(dates)-1)]
        p75_date = dates[min(p75_day, len(dates)-1)]
    else:
        med_day = p25_day = p75_day = None
        med_date = p25_date = p75_date = None

    return {'crossings': crossings, 'pct_converge': pct_converge,
            'med_day': med_day, 'med_date': med_date,
            'p25_date': p25_date, 'p75_date': p75_date}


print('OU functions defined.')


## 8. Run OU Fit and Simulate — All 4 Pairs

In [ ]:
ou_results = {}
N_PATHS = 5000

print(f'Training: Jan 2000 → {CUTOFF.date()}')
print(f'Testing:  {CUTOFF.date()} → {max_date.date()}')
print('='*60)

for a1, a2, label in PAIRS:
    p     = pair_info[(a1, a2)]['prices']
    beta  = pair_info[(a1, a2)]['beta']

    # Split prices into train/test
    train = p.loc[:CUTOFF]
    test  = p.loc[CUTOFF:]

    # Compute spread using beta estimated on TRAIN only
    X_tr  = sm.add_constant(train[a1])
    beta_tr = sm.OLS(train[a2], X_tr).fit().params[a1]

    spread_train = train[a2] - beta_tr * train[a1]
    spread_test  = test[a2]  - beta_tr * test[a1]
    spread_full  = pd.concat([spread_train, spread_test])

    # Fit OU on training spread only
    ou_params = fit_ou(spread_train)

    # Starting point = last value in training period
    S0     = spread_train.iloc[-1]
    n_test = len(spread_test)

    # Simulate forward for exactly n_test steps
    paths  = simulate_ou(ou_params, S0, n_steps=n_test, n_paths=N_PATHS)

    # Forecast dates (business days starting from first test date)
    forecast_dates = pd.bdate_range(start=spread_test.index[0], periods=n_test + 1)

    # Percentile bands
    pcts = {q: np.percentile(paths, q, axis=1) for q in [5, 10, 25, 50, 75, 90, 95]}

    # Convergence statistics
    conv = convergence_stats(paths, S0, ou_params['theta'], forecast_dates)

    # Coverage metrics: how many actual spread values fall within CI bands?
    actual_arr = spread_test.values
    # Align actual to forecast array (skip index 0 = S0, which is training end)
    n_align = min(len(actual_arr), n_test)
    in_50 = np.mean((actual_arr[:n_align] >= pcts[25][1:n_align+1]) &
                    (actual_arr[:n_align] <= pcts[75][1:n_align+1])) * 100
    in_90 = np.mean((actual_arr[:n_align] >= pcts[5][1:n_align+1]) &
                    (actual_arr[:n_align] <= pcts[95][1:n_align+1])) * 100

    # RMSE of median forecast vs actual
    rmse = np.sqrt(np.mean((pcts[50][1:n_align+1] - actual_arr[:n_align])**2))

    # Did actual spread cross theta during test period?
    theta = ou_params['theta']
    direction = 1 if S0 > theta else -1
    if direction == 1:
        actual_crossings = spread_test[spread_test <= theta]
    else:
        actual_crossings = spread_test[spread_test >= theta]
    actual_conv_date = actual_crossings.index[0] if len(actual_crossings) else None

    ou_results[(a1, a2)] = {
        'label': label, 'beta_tr': round(beta_tr, 4),
        'ou_params': ou_params, 'S0': S0,
        'spread_train': spread_train, 'spread_test': spread_test,
        'paths': paths, 'pcts': pcts, 'forecast_dates': forecast_dates,
        'conv': conv, 'in_50': in_50, 'in_90': in_90, 'rmse': rmse,
        'actual_conv_date': actual_conv_date
    }

    print(f'\n[{label}]  β_train={beta_tr:.4f}')
    print(f'  OU params:  κ={ou_params["kappa"]:.4f}  θ={ou_params["theta"]:.2f}  '
          f'σ={ou_params["sigma"]:.4f}  Half-life={ou_params["half_life"]:.1f} days')
    print(f'  S0 (last train spread): {S0:.3f}  |  θ={ou_params["theta"]:.3f}')
    print(f'  Predicted convergence:  median={conv["med_date"]}  '
          f'({conv["pct_converge"]:.1f}% paths converge in test window)')
    print(f'  Actual convergence:     {actual_conv_date.date() if actual_conv_date else "Did not cross θ in test period"}')
    print(f'  Coverage 50% CI:  {in_50:.1f}%  (ideal ~50%)')
    print(f'  Coverage 90% CI:  {in_90:.1f}%  (ideal ~90%)')
    print(f'  Median forecast RMSE:   {rmse:.4f}')


## 9. Fan Chart — Forecast vs Actual (All 4 Pairs)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

for ax, (a1, a2, label) in zip(axes, PAIRS):
    r     = ou_results[(a1, a2)]
    ou    = r['ou_params']
    pcts  = r['pcts']
    fd    = r['forecast_dates']
    S0    = r['S0']
    theta = ou['theta']

    # --- Historical training spread (last 18 months for readability) ---
    hist = r['spread_train'].loc['2021':]
    ax.plot(hist.index, hist.values, color='black', lw=1.3,
            label='Training spread (historical)', zorder=5)

    # --- Confidence bands ---
    ax.fill_between(fd, pcts[5],  pcts[95], alpha=0.12, color='royalblue', label='90% CI')
    ax.fill_between(fd, pcts[25], pcts[75], alpha=0.25, color='royalblue', label='50% CI')
    ax.plot(fd, pcts[50], color='royalblue', lw=1.8, ls='--', label='Median forecast', zorder=4)

    # --- Long-run mean θ ---
    ax.axhline(theta, color='green', ls=':', lw=1.5,
               label=f'Long-run mean θ = {theta:.2f}')

    # --- S0 marker ---
    ax.scatter([fd[0]], [S0], color='royalblue', s=80, zorder=6)

    # --- ACTUAL test spread overlaid in red ---
    actual = r['spread_test']
    ax.plot(actual.index, actual.values, color='crimson', lw=2.0,
            label='Actual spread (test period)', zorder=7)

    # --- Predicted convergence date ---
    conv = r['conv']
    if conv['med_date'] is not None:
        ax.axvline(conv['med_date'], color='orange', ls='--', lw=1.5,
                   label=f'Predicted conv. {conv["med_date"].strftime("%b %d %Y")}')

    # --- Actual convergence date ---
    if r['actual_conv_date'] is not None:
        ax.axvline(r['actual_conv_date'], color='red', ls='-', lw=2.0,
                   label=f'Actual conv. {r["actual_conv_date"].strftime("%b %d %Y")}')

    # --- Train/test boundary ---
    ax.axvline(CUTOFF, color='gray', ls=':', lw=1.2, label='Train | Test split')
    ax.text(CUTOFF, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -1,
            ' Train', fontsize=8, color='gray', va='bottom')

    ax.set_title(
        f'{label}\n'
        f'κ={ou["kappa"]:.4f}  θ={theta:.2f}  σ={ou["sigma"]:.4f}  HL={ou["half_life"]:.1f}d\n'
        f'50% CI coverage={r["in_50"]:.0f}%  90% CI coverage={r["in_90"]:.0f}%  '
        f'RMSE={r["rmse"]:.3f}',
        fontsize=9, fontweight='bold'
    )
    ax.set_ylabel('Spread (Y − β×X)')
    ax.legend(fontsize=7, loc='best')

plt.suptitle(
    'OU Forecasting — Train on Jan 2000→Feb 2023 | Test on Feb 2023→Aug 2023\n'
    'Blue bands = forecast CI  |  Red line = what actually happened',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

## 10. Detailed Single-Pair View — Best Pair (Gas Oil / ULS Diesel)

In [ ]:
# Detailed breakdown for the strongest pair
best = ('LOW SULPHUR GAS OIL', 'ULS DIESEL')
r    = ou_results[best]
ou   = r['ou_params']
pcts = r['pcts']
fd   = r['forecast_dates']

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=False)
fig.suptitle(f'Detailed OU Forecast: {best[0]} / {best[1]}', fontsize=14, fontweight='bold')

# ── Panel 1: Full training history + forecast ───────────────────────────
ax = axes[0]
ax.plot(r['spread_train'].index, r['spread_train'].values,
        color='black', lw=0.8, label='Training spread (2000–2023)', alpha=0.7)
ax.fill_between(fd, pcts[5],  pcts[95], alpha=0.12, color='royalblue')
ax.fill_between(fd, pcts[25], pcts[75], alpha=0.25, color='royalblue')
ax.plot(fd, pcts[50], color='royalblue', lw=1.5, ls='--', label='Median forecast')
ax.plot(r['spread_test'].index, r['spread_test'].values,
        color='crimson', lw=2.0, label='Actual test spread')
ax.axhline(ou['theta'], color='green', ls=':', lw=1.2, label=f'θ = {ou["theta"]:.2f}')
ax.axvline(CUTOFF, color='gray', ls=':', lw=1.0)
ax.set_title('Full History + Forecast (2000–2023 train, Feb–Aug 2023 test)')
ax.set_ylabel('Spread'); ax.legend(fontsize=8)

# ── Panel 2: Zoomed into forecast window only ───────────────────────────
ax = axes[1]
ax.fill_between(fd, pcts[5],  pcts[95], alpha=0.12, color='royalblue', label='90% CI')
ax.fill_between(fd, pcts[10], pcts[90], alpha=0.15, color='royalblue', label='80% CI')
ax.fill_between(fd, pcts[25], pcts[75], alpha=0.25, color='royalblue', label='50% CI')
ax.plot(fd, pcts[50], color='royalblue', lw=2.0, ls='--', label='Median forecast')
ax.plot(r['spread_test'].index, r['spread_test'].values,
        color='crimson', lw=2.5, label='Actual spread', zorder=5)
ax.scatter([fd[0]], [r['S0']], color='black', s=100, zorder=6, label=f'S₀ = {r["S0"]:.2f}')
ax.axhline(ou['theta'], color='green', ls=':', lw=1.5, label=f'θ = {ou["theta"]:.2f}')
conv = r['conv']
if conv['med_date']:
    ax.axvline(conv['med_date'], color='orange', ls='--', lw=1.8,
               label=f'Predicted conv. {conv["med_date"].strftime("%b %d")}')
if r['actual_conv_date']:
    ax.axvline(r['actual_conv_date'], color='red', ls='-', lw=2.0,
               label=f'Actual conv. {r["actual_conv_date"].strftime("%b %d")}')
ax.set_title(f'Zoomed: Forecast Window Only  |  '
             f'50% coverage={r["in_50"]:.0f}%  90% coverage={r["in_90"]:.0f}%  '
             f'RMSE={r["rmse"]:.3f}')
ax.set_ylabel('Spread'); ax.legend(fontsize=8, loc='best')

# ── Panel 3: Distribution of convergence times ──────────────────────────
ax = axes[2]
crossing_days = r['conv']['crossings']
if crossing_days:
    ax.hist(crossing_days, bins=40, color='royalblue', edgecolor='white',
            density=True, alpha=0.7, label='Simulated convergence days')
    ax.axvline(np.median(crossing_days), color='orange', lw=2.0, ls='--',
               label=f'Median: day {int(np.median(crossing_days))}')
    ax.axvline(np.percentile(crossing_days, 25), color='green', lw=1.5, ls=':',
               label=f'25th pct: day {int(np.percentile(crossing_days,25))}')
    ax.axvline(np.percentile(crossing_days, 75), color='red', lw=1.5, ls=':',
               label=f'75th pct: day {int(np.percentile(crossing_days,75))}')
    if r['actual_conv_date']:
        actual_day = (r['actual_conv_date'] - r['spread_test'].index[0]).days
        ax.axvline(actual_day, color='crimson', lw=2.5,
                   label=f'Actual crossing: day ~{actual_day}')
    ax.set_xlabel('Trading days until spread crosses θ')
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of First-Crossing Times across {N_PATHS:,} Simulated Paths\n'
                 f'{r["conv"]["pct_converge"]:.1f}% of paths converged within the 130-day test window')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No convergence within test window\nfor any simulated path',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)

plt.tight_layout()
plt.show()

## 11. Evaluation Summary — All Pairs

In [ ]:
print('='*80)
print('OU FORECAST EVALUATION — Train: Jan 2000→Feb 2023 | Test: Feb 2023→Aug 2023')
print('='*80)

eval_rows = []
for a1, a2, label in PAIRS:
    r  = ou_results[(a1, a2)]
    ou = r['ou_params']

    print(f'\n┌─ {label}: {a1} / {a2}')
    print(f'│  β (train):          {r["beta_tr"]}')
    print(f'│  OU κ (reversion):   {ou["kappa"]:.4f}  →  Half-life = {ou["half_life"]:.1f} trading days')
    print(f'│  OU θ (long-run μ):  {ou["theta"]:.3f}')
    print(f'│  OU σ (volatility):  {ou["sigma"]:.4f}')
    print(f'│  S₀ (Feb 2023):      {r["S0"]:.3f}  '
          f'({'above' if r["S0"] > ou["theta"] else 'below'} θ by {abs(r["S0"]-ou["theta"]):.3f})')
    print(f'│  Predicted conv:     {r["conv"]["med_date"]}  '
          f'({r["conv"]["pct_converge"]:.1f}% paths converge within test window)')
    print(f'│  Actual conv:        '
          f'{r["actual_conv_date"].date() if r["actual_conv_date"] else "Did not cross θ"}')
    match = '✓' if r['actual_conv_date'] else '✗'
    print(f'│  Convergence match:  {match}')
    print(f'│  Coverage 50% CI:    {r["in_50"]:.1f}%  (calibrated = 50%)')
    print(f'│  Coverage 90% CI:    {r["in_90"]:.1f}%  (calibrated = 90%)')
    print(f'│  Median RMSE:        {r["rmse"]:.4f}')
    print(f'└' + '─'*65)

    eval_rows.append({
        'Pair': label,
        'Half-life (days)': round(ou['half_life'], 1),
        'Predicted conv.': str(r['conv']['med_date'].date()) if r['conv']['med_date'] else 'N/A',
        'Actual conv.': str(r['actual_conv_date'].date()) if r['actual_conv_date'] else 'No crossing',
        '50% CI coverage': f"{r['in_50']:.0f}%",
        '90% CI coverage': f"{r['in_90']:.0f}%",
        'Median RMSE': round(r['rmse'], 4)
    })

eval_df = pd.DataFrame(eval_rows)
print('\n=== SUMMARY TABLE ===')
print(eval_df.to_string(index=False))


## 12. How to Interpret the Results

### Coverage Metrics

Coverage tells you whether the model's uncertainty is **well-calibrated**:

- **50% CI coverage ≈ 50%** → the model's uncertainty estimate is correct. If you see 80%, the 50% band is too wide (over-confident in uncertainty). If you see 20%, the band is too narrow (under-estimates uncertainty).

- **90% CI coverage ≈ 90%** → similarly, if the actual spread stayed inside the 90% band ~90% of the time, the model correctly captured the range of possible outcomes.

Coverage below the target means the model under-estimates spread volatility — in practice, the spread moves more than the OU model expects. This is common for commodity spreads which have fat tails and occasional regime shifts.

### RMSE Interpretation

RMSE of the median forecast vs actual spread. Because the OU model's median prediction decays toward θ quickly (for fast-reverting pairs), the RMSE tells you how far off the 'expected path' was. For slow pairs (Soybeans/Meal, Corn/Wheat), the median forecast is simply θ — so RMSE ≈ the standard deviation of the spread itself, which is the naive baseline.

### Convergence Prediction Accuracy

The key binary question: **did the actual spread cross θ during the test window?**
And if so, was the predicted date close to the actual date?

Note that for fast-reverting pairs (HL < 10 days), the spread may cross θ multiple times during the 130-day test window. The model predicts the *first* crossing — which is what matters for entering a mean-reversion trade.

### Granger Causality + OU Together

These two analyses complement each other:
- **Granger causality** tells you *which* asset moves first and by how much lag
- **OU forecasting** tells you *when* the spread will revert

Used together: when the price leader (identified by Granger) makes a large move that pushes the Z-score past ±2, enter the trade knowing the OU model predicts reversion within approximately `half_life` trading days.
